In [1]:
import os, re, json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import requests
import chromadb

try:
    from tqdm import tqdm
except Exception:
    tqdm = None


import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string


@dataclass
class Settings:
    # Ollama
    OLLAMA_URL: str = os.getenv("OLLAMA_URL", "http://localhost:11434")
    CHAT_MODEL: str = os.getenv("CHAT_MODEL", "llama3.1:8b")
    # Prefer a common embedding model mentioned in prereqs:
    EMBED_MODEL: str = os.getenv("EMBED_MODEL", "embeddinggemma:latest")

    # Chunking
    WORDS_PER_CHUNK: int = int(os.getenv("WORDS_PER_CHUNK", "300"))
    OVERLAP_WORDS: int = int(os.getenv("OVERLAP_WORDS", "60"))

    # Retrieval
    TOPK: int = int(os.getenv("TOPK", "5"))

    # Storage
    CORPUS_DIR: str = os.getenv("CORPUS_DIR", "corpus_jupyter")
    ARTIFACTS_DIR: str = os.getenv("ARTIFACTS_DIR", "rag_artifacts")
    CHROMA_PATH: str = os.getenv("CHROMA_PATH", "./chroma_db")
    CHROMA_COLLECTION: str = os.getenv("CHROMA_COLLECTION", "gutenberg_demo")
    CHROMA_DISTANCE: str = os.getenv("CHROMA_DISTANCE", "cosine")  # hnsw space

    # Workflow toggles
    DOWNLOAD_FROM_WEB: bool = (os.getenv("DOWNLOAD_FROM_WEB", "true").lower() == "true")
    FORCE_REBUILD: bool = (os.getenv("FORCE_REBUILD", "false").lower() == "true")

    # Performance
    EMBED_BATCH_SIZE: int = int(os.getenv("EMBED_BATCH_SIZE", "64"))

S = Settings()
S

Settings(OLLAMA_URL='http://localhost:11434', CHAT_MODEL='llama3.1:8b', EMBED_MODEL='embeddinggemma:latest', WORDS_PER_CHUNK=300, OVERLAP_WORDS=60, TOPK=5, CORPUS_DIR='corpus_jupyter', ARTIFACTS_DIR='rag_artifacts', CHROMA_PATH='./chroma_db', CHROMA_COLLECTION='gutenberg_demo', CHROMA_DISTANCE='cosine', DOWNLOAD_FROM_WEB=True, FORCE_REBUILD=False, EMBED_BATCH_SIZE=64)

In [2]:
BOOKS: Dict[str, str] = {
    #"Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    #"Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    #"Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    #"Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    #"Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    #"The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    #"Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    #"War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    #"Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    #"The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    #"Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    #"Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",
}

# For live demos, keep it small:
#DEFAULT_TITLES = ["Pride and Prejudice", "Frankenstein", "Adventures of Sherlock Holmes", "Jane Eyre"]
#BOOKS_TO_USE = [(t, BOOKS[t]) for t in DEFAULT_TITLES if t in BOOKS]

# If you want everything:
BOOKS_TO_USE = list(BOOKS.items())

BOOKS_TO_USE

[('A Tale of Two Cities', 'https://www.gutenberg.org/files/98/98-0.txt')]

In [3]:
# --------- Gutenberg boilerplate stripping ----------
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

def strip_gutenberg_boilerplate(txt):
    m1 = START_MARK.search(txt or "")
    m2 = END_MARK.search(txt or "")
    if m1 and m2 and m2.start() > m1.end():
        return (txt[m1.end():m2.start()] or "").strip()
    return (txt or "").strip()

def slugify(s):
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9\s-]", "", s)
    s = re.sub(r"[\s_-]+", "-", s).strip("-")
    return s or "doc"

def word_chunks(text, max_words, overlap):
    # Simple word-based chunking with overlap (deterministic + easy to explain).
    text = re.sub(r"\s+", " ", (text or "")).strip()
    words = text.split()
    if not words:
        return []
    step = max(1, max_words - overlap)
    out = []
    for start in range(0, len(words), step):
        end = start + max_words
        seg = words[start:end]
        # avoid tiny tail chunks (keeps context quality in demo)
        if len(seg) < max(60, max_words // 4):
            break
        out.append(" ".join(seg))
        if end >= len(words):
            break
    return out

def l2_normalize(mat):
    # L2-normalize embeddings. Even with cosine distance, normalization is stable
    # and makes cosine ~ dot product on unit vectors (nice for teaching).
    if mat.size == 0:
        return mat
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

In [4]:
# --------- Ollama calls ----------
session = requests.Session()
session.trust_env = False  # ignore proxy env vars (important for localhost)

def ollama_list_models():
    r = session.get(f"{S.OLLAMA_URL.rstrip('/')}/api/tags", timeout=30)
    r.raise_for_status()
    data = r.json()
    return [m.get("name", "") for m in (data.get("models") or [])]

def ollama_ready_check():
    models = ollama_list_models()
    print(f"Ollama reachable. {len(models)} model(s) found.")
    if S.CHAT_MODEL not in models:
        print(f"Warning: chat model '{S.CHAT_MODEL}' not in `ollama list`.")
    if S.EMBED_MODEL not in models:
        print(f"Warning: embed model '{S.EMBED_MODEL}' not in `ollama list`.")

def ollama_embed(texts):
    if not texts:
        return []
    r = session.post(
        f"{S.OLLAMA_URL.rstrip('/')}/api/embed",
        json={"model": S.EMBED_MODEL, "input": texts},
        timeout=600,
    )
    r.raise_for_status()
    data = r.json()
    embs = data.get("embeddings")
    if not embs:
        raise RuntimeError(f"Missing 'embeddings' in /api/embed response: {data}")
    return embs

def ollama_chat(messages, temperature=0.2, tools=None):
    payload = {
        "model": S.CHAT_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": float(temperature)},
    }
    if tools:
        payload["tools"] = tools
    r = session.post(f"{S.OLLAMA_URL.rstrip('/')}/api/chat", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()

ollama_ready_check()

Ollama reachable. 4 model(s) found.


In [6]:
# --------- paths ----------
CORPUS_DIR = Path(S.CORPUS_DIR); CORPUS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR = Path(S.ARTIFACTS_DIR); ART_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = ART_DIR / "manifest.json"
CHUNKS_PATH = ART_DIR / "chunks.json"
EMBS_PATH = ART_DIR / "embeddings.npy"

# --------- chroma ----------
chroma_client = chromadb.PersistentClient(path=S.CHROMA_PATH)

def chroma_collection(recreate=False):
    if recreate:
        try:
            chroma_client.delete_collection(S.CHROMA_COLLECTION)
        except Exception:
            pass
    return chroma_client.get_or_create_collection(
        name=S.CHROMA_COLLECTION,
        metadata={"hnsw:space": S.CHROMA_DISTANCE},
    )

def chroma_count():
    return chroma_collection().count()

In [7]:
# --------- corpus download/load ----------
def corpus_file_path(title):
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", title).strip("_")
    return CORPUS_DIR / f"{safe}.txt"

def load_docs(books):
    docs = []
    for title, url in books:
        path = corpus_file_path(title)
        if not path.exists():
            if not S.DOWNLOAD_FROM_WEB:
                continue
            print(f"Downloading: {title}")
            r = session.get(url, timeout=180)
            r.raise_for_status()
            clean = strip_gutenberg_boilerplate(r.text)
            path.write_text(clean, encoding="utf-8")
        text = path.read_text(encoding="utf-8", errors="ignore")
        docs.append({"title": title, "url": url, "path": str(path), "text": text})
    if not docs:
        raise RuntimeError("No documents loaded. Enable DOWNLOAD_FROM_WEB or place .txt files in CORPUS_DIR.")
    return docs

In [8]:
# --------- chunk + embed ----------
def build_chunks(docs):
    chunks = []
    for d in docs:
        doc_id = slugify(d["title"])
        parts = word_chunks(d["text"], S.WORDS_PER_CHUNK, S.OVERLAP_WORDS)
        for i, piece in enumerate(parts):
            chunks.append({
                "id": f"{doc_id}#{i}",
                "doc_id": doc_id,
                "chunk_index": i,
                "title": d["title"],
                "source": d["url"],
                "text": piece,
            })
    return chunks

def embed_chunks(chunks):
    bs = max(1, int(S.EMBED_BATCH_SIZE))
    mats = []
    it = range(0, len(chunks), bs)
    if tqdm:
        it = tqdm(it, desc=f"Embedding ({S.EMBED_MODEL})", total=(len(chunks)+bs-1)//bs)

    for start in it:
        batch_texts = [c["text"] for c in chunks[start:start+bs]]
        embs = ollama_embed(batch_texts)
        mats.append(np.asarray(embs, dtype=np.float32))

    mat = np.vstack(mats) if mats else np.zeros((0, 0), dtype=np.float32)
    return l2_normalize(mat)

In [9]:
# --------- artifacts ----------
def save_artifacts(docs, chunks, emb):
    CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False), encoding="utf-8")
    np.save(EMBS_PATH, emb.astype(np.float32))

    # per-doc chunk counts (nice for demos)
    by_doc = {}
    for c in chunks:
        by_doc.setdefault(c["doc_id"], 0)
        by_doc[c["doc_id"]] += 1

    doc_meta = []
    for d in docs:
        doc_id = slugify(d["title"])
        doc_meta.append({
            "title": d["title"],
            "doc_id": doc_id,
            "url": d["url"],
            "n_chunks": by_doc.get(doc_id, 0),
        })

    manifest = {
        "chunk_count": len(chunks),
        "embed_dim": int(emb.shape[1]) if emb.ndim == 2 and emb.shape[0] > 0 else None,
        "embed_model": S.EMBED_MODEL,
        "chat_model": S.CHAT_MODEL,
        "chunking": {"words_per_chunk": S.WORDS_PER_CHUNK, "overlap_words": S.OVERLAP_WORDS},
        "books_indexed": [slugify(t) for t, _ in BOOKS_TO_USE],
        "docs": doc_meta,
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

def load_artifacts():
    if not (CHUNKS_PATH.exists() and EMBS_PATH.exists() and MANIFEST_PATH.exists()):
        return None
    chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
    emb = np.load(str(EMBS_PATH))
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if len(chunks) != emb.shape[0]:
        raise RuntimeError(f"Artifact mismatch: {len(chunks)} chunks vs {emb.shape}")
    return manifest, chunks, emb

def wipe_all():
    # chroma + artifacts
    try:
        chroma_client.delete_collection(S.CHROMA_COLLECTION)
    except Exception:
        pass
    for p in [MANIFEST_PATH, CHUNKS_PATH, EMBS_PATH]:
        if p.exists():
            p.unlink()
    print("Wiped: chroma + artifacts")

In [10]:
# --------- build or restore ----------
def index_add(chunks, emb, recreate=True, batch_size=2048):
    coll = chroma_collection(recreate=recreate)
    ids = [c["id"] for c in chunks]
    docs = [c["text"] for c in chunks]
    metas = [{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"], "title": c["title"], "source": c["source"]} for c in chunks]

    for start in range(0, len(ids), batch_size):
        coll.add(
            ids=ids[start:start+batch_size],
            documents=docs[start:start+batch_size],
            metadatas=metas[start:start+batch_size],
            embeddings=emb[start:start+batch_size].tolist(),
        )

def cache_valid():
    if not MANIFEST_PATH.exists():
        return False
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

    # Validate important parameters so we don't silently reuse stale caches
    if manifest.get("embed_model") != S.EMBED_MODEL:
        return False
    if manifest.get("chunking") != {"words_per_chunk": S.WORDS_PER_CHUNK, "overlap_words": S.OVERLAP_WORDS}:
        return False
    if manifest.get("books_indexed") != [slugify(t) for t, _ in BOOKS_TO_USE]:
        return False

    expected = int(manifest.get("chunk_count") or 0)
    return expected > 0 and chroma_count() == expected

def ensure_index(books_to_use):
    if S.FORCE_REBUILD:
        print("FORCE_REBUILD=True -> rebuilding.")
        wipe_all()

    if cache_valid():
        manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8")) if CHUNKS_PATH.exists() else []
        print(f"Index ready: {chroma_count()} chunks (cache valid).")
        return manifest, chunks

    art = load_artifacts()
    if art is not None:
        manifest, chunks, emb = art
        if manifest.get("embed_model") != S.EMBED_MODEL or manifest.get("chunking") != {"words_per_chunk": S.WORDS_PER_CHUNK, "overlap_words": S.OVERLAP_WORDS}:
            print("Artifacts exist but settings changed -> rebuilding.")
        else:
            print("Restoring Chroma from artifacts...")
            index_add(chunks, emb, recreate=True)
            print(f"Restored: {chroma_count()} chunks.")
            return manifest, chunks

    print("Building from scratch...")
    docs = load_docs(books_to_use)
    chunks = build_chunks(docs)
    print(f"Built {len(chunks)} chunks. Embedding...")
    emb = embed_chunks(chunks)
    index_add(chunks, emb, recreate=True)
    save_artifacts(docs, chunks, emb)
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"Ready: {chroma_count()} chunks indexed.")
    return manifest, chunks

manifest, chunks_cache = ensure_index(BOOKS_TO_USE)
print(json.dumps(manifest, indent=2))

Building from scratch...
Downloading: A Tale of Two Cities
Built 566 chunks. Embedding...


Embedding (embeddinggemma:latest): 100%|██████████| 9/9 [10:25<00:00, 69.53s/it]


Ready: 566 chunks indexed.
{
  "chunk_count": 566,
  "embed_dim": 768,
  "embed_model": "embeddinggemma:latest",
  "chat_model": "llama3.1:8b",
  "chunking": {
    "words_per_chunk": 300,
    "overlap_words": 60
  },
  "books_indexed": [
    "a-tale-of-two-cities"
  ],
  "docs": [
    {
      "title": "A Tale of Two Cities",
      "doc_id": "a-tale-of-two-cities",
      "url": "https://www.gutenberg.org/files/98/98-0.txt",
      "n_chunks": 566
    }
  ]
}


In [11]:
def retrieve(query, k=None):
    k = int(k or S.TOPK)

    q = np.asarray(ollama_embed([query])[0], dtype=np.float32)[None, :]
    q = l2_normalize(q)[0].tolist()

    res = chroma_collection().query(
        query_embeddings=[q],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    docs = (res.get("documents") or [[]])[0]
    metas = (res.get("metadatas") or [[]])[0]
    dists = (res.get("distances") or [[]])[0]

    hits = []
    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        hits.append({
            "text": doc or "",
            "doc_id": meta.get("doc_id", "unknown"),
            "chunk_index": meta.get("chunk_index", -1),
            "title": meta.get("title", ""),
            "source": meta.get("source", ""),
            "distance": float(dist) if dist is not None else None,
            "cite": f"[{meta.get('doc_id','unknown')}#{meta.get('chunk_index',-1)}]",
        })
    return hits

def build_context(hits):
    if not hits:
        return "NO_CONTEXT"
    blocks = []
    for h in hits:
        blocks.append(f"{h['cite']} {h['title']}\n{h['text']}\n---\n")
    return "".join(blocks)

Tools

In [12]:
def tool_list_books():
    books = [{"title": t, "doc_id": slugify(t), "url": u} for t, u in BOOKS_TO_USE]
    return {"books": books}

def tool_rag_retrieve(query, k=5):
    hits = retrieve(query, k=int(k))
    return {"query": query, "k": int(k), "hits": hits}

def tool_character_context(name, k=5):
    q = f"{name} character description actions relationships"
    return tool_rag_retrieve(q, k=int(k))

def tool_quote_search(query, topn=5):
    """
    Lexical search: exact phrase matching across cached chunks.
    Demo note: deterministic first-hit scan (not ranked).
    """
    query = (query or "").strip()
    topn = int(topn)
    if not query:
        return {"query": query, "matches": []}

    matches = []
    ql = query.lower()

    def has_match(txt):
        return ql in (txt or "").lower()

    def find_span(txt):
        m = re.search(re.escape(query), (txt or ""), re.I)
        return (m.start(), m.end()) if m else None

    for c in chunks_cache:
        txt = c.get("text") or ""
        if has_match(txt):
            span = find_span(txt)
            if span:
                a, b = span
                start = max(0, a - 200); end = min(len(txt), b + 200)
                snippet = txt[start:end]
                matches.append({
                    "doc_id": c.get("doc_id"),
                    "chunk_index": c.get("chunk_index"),
                    "title": c.get("title"),
                    "source": c.get("source"),
                    "snippet": snippet,
                    "cite": f"[{c.get('doc_id')}#{c.get('chunk_index')}]",
                })
                if len(matches) >= topn:
                    break

    return {"query": query, "topn": topn, "matches": matches}

def tool_book_stats(title):
    title = (title or "").strip()
    if not title:
        return {"error": "title is required"}

    target = slugify(title)
    docs = manifest.get("docs") or []

    doc = next((d for d in docs if slugify(d.get("title","")) == target), None)
    if not doc:
        low = title.lower()
        doc = next((d for d in docs if low in (d.get("title","").lower())), None)

    return {
        "title": title,
        "doc_id": slugify(doc["title"]) if doc else target,
        "in_manifest": bool(doc),
        "manifest_doc": doc,
        "chroma_count": int(chroma_count()),
        "artifact_chunk_count": manifest.get("chunk_count"),
        "books_indexed": manifest.get("books_indexed"),
    }

TOOL_IMPL = {
    "list_books": lambda **kw: tool_list_books(),
    "quote_search": lambda **kw: tool_quote_search(**kw),
    "character_context": lambda **kw: tool_character_context(**kw),
    "rag_retrieve": lambda **kw: tool_rag_retrieve(**kw),
    "book_stats": lambda **kw: tool_book_stats(**kw),
}

Router LLM

In [13]:
# Tool schemas: only for documenting in router prompt.
pseudo_TOOL_SCHEMAS = [
    {"name": "list_books", "description": "List available Gutenberg books in the local corpus.", "params": {}},
    {"name": "quote_search", "description": "Exact phrase search; returns snippets with citations.", "params": {"query": "string", "topn": "int"}},
    {"name": "character_context", "description": "Retrieve top-k semantic chunks about a character.", "params": {"name": "string", "k": "int"}},
    {"name": "rag_retrieve", "description": "Semantic retrieval; returns hits with citations.", "params": {"query": "string", "k": "int"}},
    {"name": "book_stats", "description": "Book metadata/stats from manifest.", "params": {"title": "string"}},
]

def run_tool(name, args):
    fn = TOOL_IMPL.get(name)
    if not fn:
        return {"error": f"Unknown tool: {name}"}
    try:
        return fn(**(args or {}))
    except TypeError as e:
        return {"error": f"Bad args for {name}: {str(e)}"}
    except Exception as e:
        return {"error": f"Tool {name} failed: {type(e).__name__}: {str(e)}"}



# 7) INTENT ROUTER (LLM JSON)


INTENTS = ["quote_search", "character_context", "compare", "rag_qa", "creative_scene", "stats"]

TOOLS_INFO = "\n".join(
    [f"- {t['name']}: {t['description']} (params: {', '.join(t['params'].keys()) or 'none'})" for t in pseudo_TOOL_SCHEMAS]
)

ROUTER_SYSTEM = (
    "You are an intent router for a local Gutenberg RAG system.\n"
    "Return ONLY a single JSON object. No markdown, no code fences, no prose.\n"
    "Use exactly this schema:\n"
    "{\"intent\": string, \"confidence\": number, \"args\": object, \"rewrite\": string}\n"
    f"Allowed intents: {json.dumps(INTENTS)}.\n\n"
    "Available tools (for planning only):\n"
    f"{TOOLS_INFO}\n\n"
    "Rules:\n"
    "- quote_search: user asks for exact line/quote/phrase.\n"
    "- character_context: user asks 'who is X' or character traits.\n"
    "- stats: metadata/count requests.\n"
    "- compare: compare two characters. Put both names in args as {\"left\": \"...\", \"right\": \"...\", \"k\": 5}.\n"
    "- creative_scene: scene/dialogue writing. Put args as {\"k\": 6} (or another k).\n"
    "- rag_qa: all other questions.\n"
    "Set confidence in [0,1]. Return args as {} if not needed.\n"
    "rewrite should be the cleaned, corrected re-formatted query.\n"
)

pretty_print(ROUTER_SYSTEM)

You are an intent router for a local Gutenberg RAG system. Return ONLY a single
JSON object. No markdown, no code fences, no prose. Use exactly this schema:
{"intent": string, "confidence": number, "args": object, "rewrite": string}
Allowed intents: ["quote_search", "character_context", "compare", "rag_qa",
"creative_scene", "stats"].  Available tools (for planning only): - list_books:
List available Gutenberg books in the local corpus. (params: none) -
quote_search: Exact phrase search; returns snippets with citations. (params:
query, topn) - character_context: Retrieve top-k semantic chunks about a
character. (params: name, k) - rag_retrieve: Semantic retrieval; returns hits
with citations. (params: query, k) - book_stats: Book metadata/stats from
manifest. (params: title)  Rules: - quote_search: user asks for exact
line/quote/phrase. - character_context: user asks 'who is X' or character
traits. - stats: metadata/count requests. - compare: compare two characters. Put
both names in a

In [14]:
def extract_first_json_object(text):
    s = (text or "").strip()
    if not s:
        raise ValueError("Empty router response")

    dec = json.JSONDecoder()
    # pure JSON
    try:
        obj, _ = dec.raw_decode(s)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # find a JSON object substring
    m = re.search(r"\{.*\}", s, flags=re.S)
    if m:
        obj = json.loads(m.group(0))
        if isinstance(obj, dict):
            return obj

    raise ValueError(f"Router did not return JSON. Raw: {s[:200]}")

def route_llm(q, temperature=0.0, debug=False):
    books = ", ".join([t for t, _ in BOOKS_TO_USE])
    messages = [
        {"role": "system", "content": ROUTER_SYSTEM},
        {"role": "user", "content": f"Query: {q}\nAvailable books: {books}"},
    ]
    
    if debug:
        print("\n" + "="*80)
        print("[DEBUG] route_llm() INPUT")
        print("="*80)
        print(f"  Query: {q}")
        print(f"  Temperature: {temperature}")
        print(f"  Books available: {len(BOOKS_TO_USE)}")
        print()
    
    out = ollama_chat(messages, temperature=temperature, tools=None)
    text = out.get("message", {}).get("content", "") or out.get("response", "")
    
    if debug:
        print("[DEBUG] LLM Raw Response (first 300 chars):")
        print(f"  {text[:300]}...")
        print()
    
    obj = extract_first_json_object(text)

    if obj.get("intent") not in INTENTS:
        raise ValueError(f"Invalid intent: {obj.get('intent')}")
    if not isinstance(obj.get("args", {}), dict):
        obj["args"] = {}
    obj.setdefault("rewrite", q)
    obj.setdefault("confidence", 0.5)
    
    if debug:
        print("[DEBUG] route_llm() OUTPUT")
        print("-"*80)
        print(f"  Intent: {obj.get('intent')}")
        print(f"  Confidence: {obj.get('confidence')}")
        print(f"  Rewrite: {obj.get('rewrite')}")
        print(f"  Args: {json.dumps(obj.get('args', {}), indent=4)}")
        print("="*80 + "\n")
    
    return obj

def route_intent(q, debug=False):
    try:
        return route_llm(q, debug=debug)
    except Exception as e:
        if debug:
            print(f"[DEBUG] route_intent() EXCEPTION: {type(e).__name__}: {e}")
        print(f"[router] failed ({type(e).__name__}: {e}). Defaulting to rag_qa.")
        return {"intent": "rag_qa", "confidence": 0.5, "args": {}, "rewrite": q}

def parse_compare_names_fallback(query):
    """
    Fallback parsing for compare queries if router doesn't provide args.
    Handles: 'A vs B', 'A versus B', 'A and B compare', etc.
    """
    q = (query or "").strip()
    if not q:
        return None

    # vs / versus
    m = re.search(r"\b(.+?)\s+(vs\.?|versus)\s+(.+?)\b", q, flags=re.I)
    if m:
        return m.group(1).strip(), m.group(3).strip()

    # "compare A and B"
    m = re.search(r"\bcompare\s+(.+?)\s+and\s+(.+?)\b", q, flags=re.I)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    return None



# quick router test
tests = [
    "Find the quote 'truth universally acknowledged' and tell me the book.",
    "Who is Elizabeth Bennet?",
    "Compare Jane Eyre vs Elizabeth Bennet.",
    "Write a scene where Sherlock interviews Victor Frankenstein.",
    "How does Victor Frankenstein create life?",
    "Who's the Mobys Dick?"
]


for q in tests:
    print(q, "->", route_intent(q))

Find the quote 'truth universally acknowledged' and tell me the book. -> {'intent': 'quote_search', 'confidence': 1.0, 'args': {'query': 'truth universally acknowledged', 'topn': 1}, 'rewrite': 'truth universally acknowledged'}
Who is Elizabeth Bennet? -> {'intent': 'character_context', 'confidence': 0.9, 'args': {'name': 'Elizabeth Bennet'}, 'rewrite': 'Elizabeth Bennet'}
Compare Jane Eyre vs Elizabeth Bennet. -> {'intent': 'compare', 'confidence': 1.0, 'args': {'left': 'Jane Eyre', 'right': 'Elizabeth Bennet', 'k': 5}, 'rewrite': 'Jane Eyre vs Elizabeth Bennet'}
Write a scene where Sherlock interviews Victor Frankenstein. -> {'intent': 'creative_scene', 'confidence': 1.0, 'args': {'k': 6}, 'rewrite': 'Sherlock interviews Victor Frankenstein'}
How does Victor Frankenstein create life? -> {'intent': 'rag_qa', 'confidence': 0.8, 'args': {}, 'rewrite': 'victor frankenstein create life'}
Who's the Mobys Dick? -> {'intent': 'character_context', 'confidence': 0.9, 'args': {'name': 'Moby Dic

In [15]:
# 8) TOOL OUTPUT -> ANSWER (single LLM call)


ANSWER_SYSTEM = (
    "You are a local Gutenberg assistant.\n"
    "You MUST answer using ONLY the TOOL OUTPUTS provided.\n"
    "If the tool outputs do not contain enough information, say you don't know based on the corpus.\n\n"
    "Grounding rules:\n"
    "1) Any direct quote must appear verbatim in tool outputs and include its [doc_id#chunk_index] citation.\n"
    "2) Any factual claim about plot/characters should be supported by citations from tool outputs.\n"
    "3) Creative writing is allowed ONLY if it does not introduce unsupported factual claims; "
    "any quotes must still be sourced from tool outputs.\n"
)

def format_tool_outputs(output_from_tool_run, max_chars_per_hit=700, debug=False):
    """
    Convert tool outputs into a readable evidence block for the LLM.
    Keeps citations visible and truncates long chunks.
    """
    if debug:
        print("\n" + "="*80)
        print("[DEBUG] format_tool_outputs() INPUT")
        print("="*80)
        print(f"  Number of tool runs: {len(output_from_tool_run)}")
        print(f"  Max chars per hit: {max_chars_per_hit}")
        for i, tr in enumerate(output_from_tool_run):
            print(f"  Tool {i+1}: {tr['tool']} with args {tr.get('args', {})}")
        print()
    
    lines = []
    for tr in output_from_tool_run:
        name = tr["tool"]
        args = tr.get("args") or {}
        out = tr.get("out") or {}

        lines.append(f"TOOL: {name}")
        lines.append(f"ARGS: {json.dumps(args, ensure_ascii=False)}")

        if "error" in out:
            lines.append(f"ERROR: {out['error']}")
            lines.append("---")
            continue

        # quote_search
        if name == "quote_search":
            matches = out.get("matches") or []
            lines.append(f"MATCHES: {len(matches)}")
            for m in matches:
                snip = (m.get("snippet") or "").strip()
                lines.append(f"{m.get('cite')} {m.get('title')}")
                lines.append(snip[:max_chars_per_hit])
                lines.append("")
            lines.append("---")
            continue

        # rag_retrieve / character_context
        if name in ("rag_retrieve", "character_context"):
            hits = out.get("hits") or []
            lines.append(f"HITS: {len(hits)}")
            for h in hits:
                txt = (h.get("text") or "").strip()
                cite = h.get("cite")
                title = h.get("title")
                lines.append(f"{cite} {title}")
                lines.append(txt[:max_chars_per_hit])
                lines.append("")
            lines.append("---")
            continue

        # book_stats
        if name == "book_stats":
            lines.append(json.dumps(out, indent=2, ensure_ascii=False))
            lines.append("---")
            continue

        # list_books
        if name == "list_books":
            lines.append(json.dumps(out, indent=2, ensure_ascii=False))
            lines.append("---")
            continue

        # default
        lines.append(json.dumps(out, indent=2, ensure_ascii=False))
        lines.append("---")

    result = "\n".join(lines)
    
    if debug:
        print("[DEBUG] format_tool_outputs() OUTPUT")
        print("-"*80)
        print(f"  Formatted output length: {len(result)} chars")
        print(f"  Preview :")
        print(f"  {result}...")
        print("="*80 + "\n")
    
    return result

def process_answer_from_tools(user_query, output_from_tool_run, temperature=0.2, debug=False):
    if debug:
        print("\n" + "="*80)
        print("[DEBUG] process_answer_from_tools() INPUT")
        print("="*80)
        print(f"  User query: {user_query}")
        print(f"  Number of tool runs: {len(output_from_tool_run)}")
        print(f" Tool runs:")
        for i, tr in enumerate(output_from_tool_run):
            print(f"  {i+1}. {tr['tool']} with args {tr.get('args', {})} and output keys {list((tr.get('out') or {}).keys())}")
        print(f"  Temperature: {temperature}")
        print()
    
    evidence = format_tool_outputs(output_from_tool_run, debug=debug)
    
    messages = [
        {"role": "system", "content": ANSWER_SYSTEM},
        {"role": "user", "content": f"USER QUERY:\n{user_query}\n\nTOOL OUTPUTS:\n{evidence}"},
    ]
    
    if debug:
        print("[DEBUG] Calling ollama_chat with prepared messages")
        print(f"  System message length: {len(ANSWER_SYSTEM)} chars")
        print(f"  User message length: {len(messages[1]['content'])} chars")
        print()
    
    out = ollama_chat(messages, temperature=temperature)
    answer = out.get("message", {}).get("content", "") or out.get("response", "")
    
    if debug:
        print("[DEBUG] process_answer_from_tools() OUTPUT")
        print("-"*80)
        print(f"  Answer length: {len(answer)} chars")
        print(f"  Preview (first 400 chars):")
        print(f"  {answer[:400]}...")
        print("="*80 + "\n")
    
    return answer

In [16]:
# 9) DISPATCH (single entry point)


def dispatch(query, trace=True, temperature_router=0.0, temperature_answer=0.2, debug=False):
    if debug:
        print("\n" + "="*80)
        print("[DEBUG] dispatch() STARTING")
        print("="*80)
        print(f"  Input query: {query}")
        print(f"  Trace enabled: {trace}")
        print(f"  Debug enabled: {debug}")
        print(f"  Router temperature: {temperature_router}")
        print(f"  Answer temperature: {temperature_answer}")
        print()
    
    route = route_intent(query, debug=debug)
    intent = route.get("intent", "rag_qa")  # {'intent': 'quote_search', 'confidence': 0.9, 'args': {'query': 'truth universally acknowledged'}, 'rewrite': 'truth universally acknowledged'}
    args = route.get("args") or {}
    q = route.get("rewrite") or query

    if debug or trace:
        print("[router]", route)
        if debug:
            print(f"  Resolved intent: {intent}")
            print(f"  Resolved query: {q}")
            print()

    output_from_tool_run = []

    # Decide which tool(s) to run (deterministic orchestration)
    if intent == "quote_search":
        tool_args = {"query": q, "topn": int(args.get("topn", 5))}
        out = run_tool("quote_search", tool_args)
        output_from_tool_run.append({"tool": "quote_search", "args": tool_args, "out": out})

    elif intent == "character_context":
        # router should provide name; fallback: use whole query
        name = args.get("name") or q
        tool_args = {"name": name, "k": int(args.get("k", S.TOPK))}
        out = run_tool("character_context", tool_args)
        output_from_tool_run.append({"tool": "character_context", "args": tool_args, "out": out})

    elif intent == "stats":
        title = args.get("title") or q
        tool_args = {"title": title}
        out = run_tool("book_stats", tool_args)
        output_from_tool_run.append({"tool": "book_stats", "args": tool_args, "out": out})

    elif intent == "compare":
        left = args.get("left")
        right = args.get("right")
        k = int(args.get("k", S.TOPK))

        if not left or not right:
            parsed = parse_compare_names_fallback(q)
            if parsed:
                left, right = parsed

        # If still missing, do a general retrieval once.
        if not left or not right:
            tool_args = {"query": q, "k": k}
            out = run_tool("rag_retrieve", tool_args)
            output_from_tool_run.append({"tool": "rag_retrieve", "args": tool_args, "out": out})
        else:
            outL = run_tool("character_context", {"name": left, "k": k})
            outR = run_tool("character_context", {"name": right, "k": k})
            output_from_tool_run.append({"tool": "character_context", "args": {"name": left, "k": k}, "out": outL})
            output_from_tool_run.append({"tool": "character_context", "args": {"name": right, "k": k}, "out": outR})

    elif intent == "creative_scene":
        # For creative scenes, we retrieve relevant context once.
        k = int(args.get("k", max(6, S.TOPK)))
        tool_args = {"query": q, "k": k}
        out = run_tool("rag_retrieve", tool_args)
        output_from_tool_run.append({"tool": "rag_retrieve", "args": tool_args, "out": out})

    else:
        # rag_qa default: just retrieve
        k = int(args.get("k", S.TOPK))
        tool_args = {"query": q, "k": k}
        out = run_tool("rag_retrieve", tool_args)
        output_from_tool_run.append({"tool": "rag_retrieve", "args": tool_args, "out": out})

    # Final answer based ONLY on tool outputs
    if debug:
        print("[DEBUG] dispatch() - Generating final answer")
        print(f"  Tool runs collected: {len(output_from_tool_run)}")
        print(f"  Tools used: {[tr['tool'] for tr in output_from_tool_run]}")
        print(f" Output from tool runs :")
        for tr in output_from_tool_run:
            print(f"   - {tr['tool']}")
            print("   - ", tr['out'])
        print()
    
    answer = process_answer_from_tools(q, output_from_tool_run, temperature=temperature_answer, debug=debug)
    
    if debug:
        print("[DEBUG] dispatch() COMPLETE")
        print("="*80 + "\n")

    return {
        "intent": intent,
        "route": route,
        "answer": answer,
        "tools_used": [{"tool": tr["tool"], "args": tr["args"]} for tr in output_from_tool_run],
        "output_from_tool_run": output_from_tool_run if trace else None,
    }

def ask(q, trace=True):
    print("Q:", q)
    out = dispatch(q, trace=trace, debug=False)
    print("\nANSWER:\n", out["answer"])
    if trace:
        print("\nTOOLS USED:")
        for t in out["tools_used"]:
            print(" -", t["tool"], t["args"])
    return out

In [17]:
ask("Find the quote 'It is a far, far better thing that I do' and tell me the book.", trace=True)

Q: Find the quote 'It is a far, far better thing that I do' and tell me the book.
[router] {'intent': 'quote_search', 'confidence': 1.0, 'args': {'query': 'It is a far, far better thing that I do', 'topn': 1}, 'rewrite': 'It is a far, far better thing that I do'}

ANSWER:
 "It is a far, far better thing that I do, than I have ever done; it is a far, far better rest that I go to than I have ever known." [a-tale-of-two-cities#565]

TOOLS USED:
 - quote_search {'query': 'It is a far, far better thing that I do', 'topn': 1}


{'intent': 'quote_search',
 'route': {'intent': 'quote_search',
  'confidence': 1.0,
  'args': {'query': 'It is a far, far better thing that I do', 'topn': 1},
  'rewrite': 'It is a far, far better thing that I do'},
 'answer': '"It is a far, far better thing that I do, than I have ever done; it is a far, far better rest that I go to than I have ever known." [a-tale-of-two-cities#565]',
 'tools_used': [{'tool': 'quote_search',
   'args': {'query': 'It is a far, far better thing that I do', 'topn': 1}}],
 'output_from_tool_run': [{'tool': 'quote_search',
   'args': {'query': 'It is a far, far better thing that I do', 'topn': 1},
   'out': {'query': 'It is a far, far better thing that I do',
    'topn': 1,
    'matches': [{'doc_id': 'a-tale-of-two-cities',
      'chunk_index': 565,
      'title': 'A Tale of Two Cities',
      'source': 'https://www.gutenberg.org/files/98/98-0.txt',
      'snippet': 'orehead that I know and golden hair, to this place--then fair to look upon, with not a tr